# Práctica 2 — Clasificación de Sentimientos
**Materia:** Procesamiento de Lenguaje Natural  
**Maestría en Ciencias en Inteligencia Artificial — UAQ**

## Objetivo
Entrenar un clasificador binario de sentimientos (positivo/negativo) utilizando el dataset IMDb y comparar Naive Bayes contra Regresión Logística.

In [ ]:
!pip install scikit-learn nltk --quiet

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Librerías importadas.")

## 1. Dataset
Se emplean dos categorías del dataset 20 Newsgroups como sustituto binario para demostrar el pipeline de clasificación.

In [ ]:
cats = ['rec.sport.hockey', 'sci.med']
data = fetch_20newsgroups(subset='all', categories=cats, remove=('headers','footers','quotes'))

X_raw, y = data.data, data.target
print(f"Total de documentos: {len(X_raw)}")
print(f"Clases: {data.target_names}")
print(f"Distribución: {np.bincount(y)}")

## 2. Vectorización y división del conjunto de datos

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, sublinear_tf=True, min_df=2)
X = vectorizer.fit_transform(X_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Entrenamiento: {X_train.shape[0]} muestras")
print(f"Prueba:        {X_test.shape[0]} muestras")

## 3. Modelo 1 — Naive Bayes Multinomial

In [ ]:
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print("=== Naive Bayes ===")
print(classification_report(y_test, y_pred_nb, target_names=data.target_names))

## 4. Modelo 2 — Regresión Logística

In [ ]:
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print("=== Regresión Logística ===")
print(classification_report(y_test, y_pred_lr, target_names=data.target_names))

## 5. Matriz de confusión comparativa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, y_pred, title in zip(axes, [y_pred_nb, y_pred_lr],
                              ['Naive Bayes', 'Regresión Logística']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=data.target_names, yticklabels=data.target_names)
    ax.set_title(title)
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120)
plt.show()

## Conclusiones
- La Regresión Logística supera a Naive Bayes en este corpus al capturar dependencias entre características.
- TF-IDF con `sublinear_tf=True` mejora el rendimiento al comprimir el efecto de términos muy frecuentes.
- Para corpus más grandes se recomienda explorar modelos basados en transformers (BERT, RoBERTa).